# Guía de Desarrollo de Herramientas (Tools)

Este notebook documenta cómo crear, probar e integrar herramientas en el dashboard EGOFET.

**Concepto:** Una herramienta es un script de Python que el dashboard ejecuta sobre un archivo adjunto (HDF5, CSV, foto) con parámetros elegidos por el usuario. Produce figuras, tablas o imágenes que se registran automáticamente en la memoria.

**Analogía del sello:** Los templates son moldes para pre-rellenar campos al crear una memoria (el sello). Las herramientas son procesos que transforman datos crudos en visualizaciones (la tinta que se aplica). Una memoria puede existir sin herramientas, pero las herramientas enriquecen las memorias con análisis.

**Flujo:**
1. Adjuntar archivo HDF5 a una memoria (References)
2. Pulsar "Apply tool" → elegir herramienta → configurar parámetros
3. La herramienta procesa el archivo y genera figuras/tablas
4. El dashboard registra automáticamente en Analysis y Figures

**Estructura:**
- Sección 1-7: Guía conceptual de herramientas
- Sección 8: Flujo legacy DB+CSV (educativo)

---

**Rutas importantes:**
- Herramientas: `src/dashboard_tools/*.py`
- Base de datos: `data/memories.db`
- Resultados: `data/processed/memory_<id>/<tool>_<stamp>/`
- Templates: `dashboard/templates/*.yaml`

## 1. ¿Qué es una Tool?

Una **Tool** es una clase Python que:
- Declara qué tipo de archivo acepta (`input_ref_type`)
- Define qué parámetros pide al usuario (`parameters`)
- Implementa un método `run()` que procesa el archivo y devuelve resultados

Las herramientas viven en `src/dashboard_tools/` y se descubren automáticamente.

**Ejemplo mínimo:**
```python
from src.dashboard_tools.base import Tool, ToolResult, ParameterSchema

class MiTool(Tool):
    name = "mi_tool"
    display_name = "Mi herramienta"
    description = "Hace algo útil"
    input_ref_type = "hdf5"
    parameters = ()
    
    def run(self, ref_path, params, output_dir):
        # Procesar ref_path
        # Guardar figuras en output_dir
        return ToolResult(figures=[...], summary={...})
```

In [ ]:
import os, sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Asegurar que el proyecto está en PYTHONPATH
PROJECT_ROOT = Path(os.getcwd()).resolve()
if 'src' not in [d.name for d in PROJECT_ROOT.iterdir()]:
    # Estamos dentro de notebooks/, subir un nivel
    PROJECT_ROOT = PROJECT_ROOT.parent
    if 'src' not in [d.name for d in PROJECT_ROOT.iterdir()]:
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src import db
from src.dashboard_tools import list_tools, get_tool
from src.dashboard_tools.base import Tool, ToolResult, ParameterSchema

db.init_db()
print('Project root:', PROJECT_ROOT)
print('Database:', db.DB_PATH)
print('Herramientas disponibles:', [t.name for t in list_tools()])

## 2. Anatomía de una Tool

Cada herramienta tiene 5 variables de clase y 1 método obligatorio:

| Atributo | Tipo | Descripción |
|----------|------|-------------|
| `name` | `str` | ID único, snake_case (ej: `"transfer_vth"`) |
| `display_name` | `str` | Nombre legible para la UI |
| `description` | `str` | Explicación que aparece en el diálogo |
| `input_ref_type` | `str` | `"hdf5"`, `"csv"`, `"photo"`, `"figure"` |
| `measurement_mode` | `str \| None` | `"transfer"`, `"output"`, o `None` |
| `parameters` | `tuple[ParameterSchema]` | Lista de parámetros que pide al usuario |

**Método `run()`:**
```python
def run(self, ref_path: Path, params: dict, output_dir: Path) -> ToolResult:
    # ref_path: ruta al archivo (HDF5, CSV, etc.)
    # params: dict con valores ya validados según el schema
    # output_dir: directorio donde guardar resultados
    
    # 1. Leer el archivo
    # 2. Procesar con params
    # 3. Guardar figuras/tablas en output_dir
    # 4. Devolver ToolResult
```

## 3. Los 7 tipos de parámetros

Cada parámetro se define con `ParameterSchema(name, label, type, default, choices, help)`.

| Tipo | Widget en UI | Ejemplo |
|------|--------------|---------|
| `"float"` | Input numérico decimal | `v_min: -0.40` |
| `"int"` | Input numérico entero | `ncols: 3` |
| `"bool"` | Switch on/off | `loop: True` |
| `"str"` | Input de texto | `title: "Mi análisis"` |
| `"float_pair"` | Dos inputs min/max | `v_min_range: (-0.40, -0.38)` |
| `"choice"` | Select dropdown | `color: "blue"` |
| `"measurements"` | Checkboxes de mediciones | Seleccionar cuáles del HDF5 |

**Ejemplo de declaración:**
```python
parameters = (
    ParameterSchema(
        name="measurements",
        label="Mediciones a procesar",
        type="measurements",
        default=(),  # Vacío = todas
        help="Ninguna selección = procesar todas las mediciones"
    ),
    ParameterSchema(
        name="v_min_range",
        label="Rango de búsqueda v_min (V)",
        type="float_pair",
        default=(-0.40, -0.38),
        help="Límite inferior de la ventana de ajuste"
    ),
    ParameterSchema(
        name="loop",
        label="Incluir sweep inverso",
        type="bool",
        default=True
    ),
)
```

## 4. ToolResult y figure_roles

El resultado de una herramienta es un `ToolResult`:

```python
@dataclass
class ToolResult:
    figures: list[Path] = field(default_factory=list)
    figure_roles: list[str] = field(default_factory=list)
    tables: list[Path] = field(default_factory=list)
    summary: dict = field(default_factory=dict)
    description: str = ""
```

**`figure_roles`** clasifica cada figura:
- `"overview"` → Se muestra grande en la carta Figures
- `"curve"` → Se muestra como miniatura, expandible
- Otros → Se tratan como figuras sueltas

**Ejemplo:**
```python
# 1 overview + 23 curvas individuales
return ToolResult(
    figures=[overview.png, curve_1.png, curve_2.png, ...],
    figure_roles=["overview"] + ["curve"] * 23,
    tables=[results.csv],
    summary={"mean": 0.034, "std": 0.069, "n_measurements": 23},
    description="V_th extraído para 23 curvas de transferencia"
)
```

## 5. Registro automático

Las herramientas se descubren automáticamente al importar `src.dashboard_tools`.

**Reglas:**
1. Cada archivo `.py` en `src/dashboard_tools/` se examina
2. Se ignoran módulos que empiezan con `_` (ej: `_common.py`)
3. Se ignoran `base.py` y `registry.py`
4. Se busca cualquier subclase de `Tool` definida en el módulo
5. Se registra bajo su atributo `name`

**Para añadir una herramienta:**
1. Crear `src/dashboard_tools/mi_herramienta.py`
2. Definir `class MiTool(Tool):` con los atributos requeridos
3. Implementar `run()`
4. Reiniciar el dashboard (o recargar módulos)

**Ver herramientas disponibles:**
```python
from src.dashboard_tools import list_tools
for tool in list_tools():
    print(f"{tool.name}: {tool.display_name} ({tool.input_ref_type})")
```

In [ ]:
# Ver herramientas disponibles
print('Herramientas disponibles:')
for tool in list_tools():
    print(f"  {tool.name:20s} | {tool.display_name:30s} | {tool.input_ref_type}")

print('\nDetalles de transfer_vth:')
t = get_tool('transfer_vth')
print(f"  measurement_mode: {t.measurement_mode}")
print(f"  parameters: {[p.name for p in t.parameters]}")

## 6. El flujo completo

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  Referencias    │────▶│  Apply tool      │────▶│  Analysis       │
│  (HDF5 adjunto) │     │  - Tool selector │     │  - Params       │
│                 │     │  - Parámetros    │     │  - Summary      │
│                 │     │  - [Run]         │     │  - CSV (opc.)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
                                 │
                                 ▼
                          ┌──────────────────┐
                          │  Figures         │
                          │  - Overview      │
                          │  - Curvas (grid) │
                          └──────────────────┘
```

**Qué hace `run_tool_on_memory()` (en `dashboard/tool_runs.py`):**
1. Valida los parámetros con `tool.coerce(params)`
2. Crea directorio `data/processed/memory_<id>/<tool>_<stamp>/`
3. Llama `tool.run(ref_path, params, output_dir)`
4. Registra `analysis_results` con `params_json={params, summary, description, source}`
5. Añade referencias `figure` con metadata `{analysis_id, tool, role}`

**Resultado en la base de datos:**
- `analysis_results`: 1 fila con parámetros y resumen
- `memory_references`: N filas de tipo `figure` (1 overview + N curvas)

## 7. Probar sin el dashboard

Puedes ejecutar herramientas directamente desde un notebook:

```python
from dashboard.tool_runs import run_tool_on_memory

# memory_id: ID de una memoria existente
# ref_path: ruta relativa del archivo (ej: "data/raw/test.hdf5")
# tool_name: nombre de la herramienta (ej: "transfer_vth")
# params: dict con parámetros (vacío = defaults)

analysis_id = run_tool_on_memory(
    memory_id=1,
    ref_path="data/raw/2026-10-02_test.hdf5",
    tool_name="transfer_vth",
    params={"v_min_range": (-0.40, -0.38), "v_max_range": (-0.06, 0.00)}
)
```

In [ ]:
# Ejemplo: ejecutar transfer_vth directamente
from dashboard.tool_runs import run_tool_on_memory, PROJECT_ROOT

# Buscar una memoria con HDF5
memories = db.list_memories()
mem_with_hdf5 = None
for m in memories:
    refs = db.get_memory(m['id'])['references']
    if any(r['ref_type'] == 'hdf5' for r in refs):
        mem_with_hdf5 = m
        break

if mem_with_hdf5:
    print(f'Usando memoria: {mem_with_hdf5["title"]}')
    
    # Obtener el primer HDF5
    refs = db.get_memory(mem_with_hdf5['id'])['references']
    hdf5_ref = next(r['ref_path'] for r in refs if r['ref_type'] == 'hdf5')
    
    print(f'Archivo: {hdf5_ref}')
    
    # Ejecutar herramienta
    analysis_id = run_tool_on_memory(
        memory_id=mem_with_hdf5['id'],
        ref_path=hdf5_ref,
        tool_name='transfer_vth',
        params={}
    )
    
    print(f'Analysis ID: {analysis_id}')
    
    # Ver resultado
    mem = db.get_memory(mem_with_hdf5['id'])
    analysis = next(a for a in mem['analysis_results'] if a['id'] == analysis_id)
    print(f'\nSummary: {analysis["params_json"][:200]}...')
    
    fig_refs = [r for r in mem['references'] if r['ref_type'] == 'figure']
    print(f'Figuras generadas: {len(fig_refs)}')
else:
    print('No hay memorias con HDF5 en la base de datos')

## 8. Ejemplo conceptual: herramienta para CSV

**Pregunta:** ¿Cómo haría una herramienta que lea CSV en vez de HDF5?

**Respuesta:** Cambiar `input_ref_type` y quitar `measurement_mode` (no aplica a CSV).

**Pseudocódigo:**
```python
class CSVDetector(Tool):
    name = "csv_detector"
    display_name = "Detectar anomalías en CSV"
    input_ref_type = "csv"
    measurement_mode = None  # No hay mediciones en CSV
    parameters = (
        ParameterSchema("threshold", "Umbral", "float", 2.0, 
                       help="Desviaciones estándar para detectar outliers"),
        ParameterSchema("columns", "Columnas a analizar", "choice", "all",
                       choices=("all", "numeric_only"))
    )
    
    def run(self, ref_path, params, output_dir):
        df = pd.read_csv(ref_path)
        # Análisis...
        # Guardar figura...
        return ToolResult(
            figures=[output_dir / "anomalies.png"],
            figure_roles=["overview"],
            summary={"n_anomalies": 5, "threshold": params["threshold"]}
        )
```

**Para que aparezca el botón "Apply tool" en referencias CSV:**
- Actualmente el botón solo aparece para HDF5 (línea 21 de `dashboard/ui/tool_dialog.py`)
- Habría que modificar para que aparezca según `input_ref_type` de cada herramienta
- Pendiente de implementar

---

## 9. Flujo legacy: DB + CSV (educativo)

**Nota:** Esta sección muestra el flujo manual que usábamos antes de las herramientas. El dashboard moderno hace esto automáticamente cuando ejecutas una herramienta, pero entender este flujo ayuda a comprender qué está pasando por debajo.

### Paso 1: Crear memoria
```python
memory_id = db.create_memory(
    title="Mi experimento",
    type="experiment",
    subtype="batch",
    date="2026-10-09",
    substrate="Kapton",
    custom_fields={"thickness": "125um"}
)
```

### Paso 2: Adjuntar archivo
```python
db.add_reference(memory_id, "hdf5", "data/raw/test.hdf5")
```

### Paso 3: Procesar y guardar CSV
```python
# Procesar con src.analysis...
df.to_csv("data/processed/results.csv")
db.add_reference(memory_id, "csv", "data/processed/results.csv")
```

### Paso 4: Registrar análisis
```python
db.save_analysis_result(
    memory_id=memory_id,
    function_name="mi_analisis",
    params={"param1": valor1},
    results_csv_path="data/processed/results.csv"
)
```

**Ahora el dashboard puede:**
- Leer la memoria y sus referencias
- Mostrar el CSV como gráfica (carta Figures)
- Mostrar parámetros del análisis (carta Analysis)

**Con herramientas, los pasos 3-4 los hace `run_tool_on_memory()` automáticamente.**

In [ ]:
# Ejemplo rápido del flujo legacy (sin crear memoria real)
print('Flujo legacy DB+CSV:')
print('1. Crear memoria → db.create_memory()')
print('2. Adjuntar HDF5 → db.add_reference(id, "hdf5", path)')
print('3. Procesar → extract_vth_for_transfers(...)')
print('4. Guardar CSV → df.to_csv("data/processed/...")')
print('5. Referenciar CSV → db.add_reference(id, "csv", csv_path)')
print('6. Registrar análisis → db.save_analysis_result(...)')
print()
print('Las herramientas hacen los pasos 3-6 automáticamente.')
print('El usuario solo ve: References → Apply tool → Figures/Analysis')